# CMsiRNAdb quality control and experimental encoding

DataCleaner drops the rows and columns that are not fit for modeling, ExperimentalEncoder does one-hot-encoding for Cell_type, and FeatureNormalizer normalizes to the range 0 to 1.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import importlib
import pandas as pd

import data_cleaning
import experimental_encoding
importlib.reload(data_cleaning)
importlib.reload(experimental_encoding)

from data_cleaning import DataCleaner
from experimental_encoding import ExperimentalEncoder, FeatureNormalizer

### Load the raw dataset



In [ ]:
raw_path = os.environ.get("CMSIRNA_RAW_DATA_PATH")
df_raw = pd.read_csv(raw_path, sep="\t", low_memory=False)

required = ["Antisense_seqence", "Sense_seqence", "length_anti_sense_strand",
            "length_sense_strand", "Concentration", "Time_of_administration",
            "Cell_Type", "Inhibition"]
missing = [c for c in required if c not in df_raw.columns]
assert not missing, f"raw data is missing expected columns: {missing}"

print("Raw shape:", df_raw.shape)
df_raw.head()

### Do the length columns match the real sequences

The dataset has a length column for each strand, and does not always agree with actual seq length. The cell below counts the characters and compares, so downstream we trust the sequence (.str.len()), not the length column.

In [41]:
# a few mismatching rows per strand, one per distinct offset, as proof
pd.set_option("display.max_colwidth", 80)

def show_mismatch_examples(df, seq_col, len_col, name):
    pairs = df.dropna(subset=[seq_col, len_col]).copy()
    pairs["actual_len"] = pairs[seq_col].astype(str).str.len()
    pairs["diff"] = pairs["actual_len"] - pairs[len_col]
    notmatch = pairs[pairs["diff"] != 0].sort_values("diff")
    one_per_offset = notmatch.groupby("diff", group_keys=False).head(1)
    cols = [c for c in ["ID", seq_col, len_col, "actual_len", "diff"] if c in pairs.columns]
    print(f"{name}: {len(notmatch)} mismatching rows; one example per offset:")
    display(one_per_offset[cols])

show_mismatch_examples(df_raw, "Antisense_seqence", "length_anti_sense_strand", "ANTISENSE")
show_mismatch_examples(df_raw, "Sense_seqence", "length_sense_strand", "SENSE")

ANTISENSE: 3271 mismatching rows; one example per offset:


,ID,Antisense_seqence,length_anti_sense_strand,actual_len,diff
40375,080-13-11-14096-30n-XXX--20.00,UCGCUGGAGGCACCAAUGA,21.0,19,-2.0
42399,088-13-01-15407-100n-48h-56.00,AUGACUGCACACUUGCUGGCCUGU,25.0,24,-1.0
17167,014-04-11-05094-XX-72h-37.35,AUGUAGAAAAGGCAUGAAGCAGUU,23.0,24,1.0
5604,004-03-02-01755-XX-XXX-38.00,UCCAUUUGGGUAGUAUUAbAb,19.0,21,2.0


SENSE: 3639 mismatching rows; one example per offset:


,ID,Sense_seqence,length_sense_strand,actual_len,diff
41841,081-13-01-15040-0.1n-24h-45.90,CUGGCACCUACGUGGUGG,21.0,18,-3.0
32617,041-10-02-10633-50n-24h-7.80,GAAGUAUAUUUUUUAUUGC,21.0,19,-2.0
28404,032-09-04-08964-0.1n-48h-24.30,GAAUGAGUGCACAGCUAAGA,21.0,20,-1.0
38568,071-12-13-13117-5m-XXX-77.00,GACCAGCUUGUUUGUGAAACA,20.0,21,1.0
8308,007-03-07-03161-20n-24h-78.73,CAGCCCCUUAUUGUUAUACGAAG,21.0,23,2.0
7437,004-03-06-02829-1m-XXX-98.40,AbGCCCCUUAUUGUUAUACGAUUAb,22.0,25,3.0
33797,044-11-30-11518-0.5m-XXX-70.30,GCUCAAC(A2N)UAUUUGAUCAGUA,21.0,25,4.0
19372,015-04-06-06128-1m-XXX-72.20,CAACGUACCCUUCAUUGAUU(invAb),21.0,27,6.0
18583,015-04-11-05492-XX-72h-66.70,UUCUACAGUGGCCUUAUCCC(invAb),20.0,27,7.0
33748,044-11-30-11519-0.5m-XXX-70.30,GCUCAAC(A2N)U(A2N)UUUGAUCAGUA,21.0,29,8.0


### Quality control with DataCleaner

DataCleaner removes the rows and columns that should not reach to the modeling, and prints a count for every drop. It drops, in order:

- in-vivo rows, where a whole animal was dosed (mg/kg doses, animal-only cell strings)
- mM concentrations, which are not a realistic siRNA dose and are almost certainly a unit typo
- unknown or RGA cell lines, and any inhibition outside the range -100 to 100, since the label is then unusable or impossible
- missing strands, or strands longer than 25 nt, with length taken from the sequence

In [ ]:
cleaner = DataCleaner(df_raw.copy())
df_clean = cleaner.clean()
print(f"\nRows: {len(df_raw)} to {len(df_clean)} (dropped {len(df_raw) - len(df_clean)})")
df_clean.head()

Checks:

In [43]:
inhibition = pd.to_numeric(df_clean["Inhibition"], errors="coerce")
sense_len = df_clean["Sense_seqence"].str.replace(r"\s", "", regex=True).str.len()
anti_len = df_clean["Antisense_seqence"].str.replace(r"\s", "", regex=True).str.len()

assert inhibition.between(-100, 100).all(), "inhibition outside [-100, 100] survived"
assert sense_len.between(1, 25).all(), "sense strand outside 1-25 nt survived"
assert anti_len.between(1, 25).all(), "antisense strand outside 1-25 nt survived"

print("Inhibition range:", round(inhibition.min(), 1), "to", round(inhibition.max(), 1))
print("Sense length:", sense_len.min(), "-", sense_len.max())
print("Antisense length:", anti_len.min(), "-", anti_len.max())
print("Cell types kept:", df_clean["Cell_Type"].nunique())
inhibition.describe().round(1)

Inhibition range: -99.5 to 100.0
Sense length: 17 - 25
Antisense length: 13 - 25
Cell types kept: 20


count    35256.0
mean        40.6
std         34.0
min        -99.5
25%         12.7
50%         41.2
75%         69.5
max        100.0
Name: Inhibition, dtype: float64

### Encoding the experimental conditions with ExperimentalEncoder

Concentration becomes nM. Rows above 200 nM are dropped.

Time of administration becomes hours. We parse "48h", "2 days" or "24-48h" (averaged) into a number of hours.

Cell type becomes a one-hot vector over a sorted list of cell lines, so the same position always means the same cell line across rows.

The cleaned table feeds the encoder and the result is saved as the file CMsiRNA_clean_encoded.tsv.

In [26]:
encoder = ExperimentalEncoder(df_clean.copy())
df_encoded = encoder.encode()
df_encoded[["Concentration", "Concentration_nM",
            "Time_of_administration", "Time_of_administration_h",
            "Cell_Type"]].head()

,Concentration,Concentration_nM,Time_of_administration,Time_of_administration_h,Cell_Type
0,100nM,100.0,48h,48.0,Hela
1,100nM,100.0,48h,48.0,Hela
2,100nM,100.0,48h,48.0,Hela
3,100nM,100.0,48h,48.0,Hela
4,100nM,100.0,48h,48.0,Hela


In [27]:
# the cleaned + encoded table
out_path = "../data/CMsiRNA_clean_encoded.tsv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
df_encoded.to_csv(out_path, sep="\t", index=False)
print("Saved", df_encoded.shape, "to", out_path)

Saved (33450, 29) -> ../data/CMsiRNA_clean_encoded.tsv


### How the time values parse

A look at every time string, how many rows use it, and the number of hours parse_time turns it into.

In [30]:
time_counts = df_encoded["Time_of_administration"].value_counts(dropna=False)
time_summary = pd.DataFrame({
    "count": time_counts,
    "parsed_hours": [cleaner.parse_time(v) for v in time_counts.index],
})
time_summary.index.name = "raw_value"

print("Distinct time values:", df_encoded["Time_of_administration"].nunique(dropna=True))
print("Missing (NaN) rows:", df_encoded["Time_of_administration"].isna().sum())
time_summary

Distinct time values: 6
Missing (NaN) rows  : 6091


,count,parsed_hours
raw_value,,
24h,19274,24.0
48h,6158,48.0
NaN,6091,NaN
72h,1369,72.0
7 days,314,168.0
40 h,199,40.0
2 days,45,48.0


### Missing time of administration

About a fifth of rows have no recorded time. 

In [ ]:
t = df_encoded["Time_of_administration_h"]
miss = t.isna()
inh = pd.to_numeric(df_encoded["Inhibition"], errors="coerce")

print(f"total rows: {len(df_encoded)}")
print(f"missing time (NaN): {miss.sum()} ({miss.mean() * 100:.1f}%)")
print(f"24h share among reported: {(t.dropna() == 24).mean() * 100:.1f}%")
print("\nreported time values (rows):")
print(t.dropna().value_counts())

# missing-time rows vs the rest (dose and knockdown)
pd.DataFrame({
    "n": [miss.sum(), (~miss).sum()],
    "median Concentration_nM": [df_encoded.loc[miss, "Concentration_nM"].median(),
                                df_encoded.loc[~miss, "Concentration_nM"].median()],
    "median Inhibition": [inh[miss].median(), inh[~miss].median()],
}, index=["time missing", "time present"]).round(2)

total rows               : 33450
missing time (NaN)       : 6091 (18.2%)
24h share among reported : 70.4%

reported time values (rows):
Time_of_administration_h
24.0     19274
48.0      6203
72.0      1369
168.0      314
40.0       199
Name: count, dtype: int64


,n,median Concentration_nM,median Inhibition
time missing,6091,10.0,51.60
time present,27359,1.0,38.41


In [ ]:
# checked by cell line and by source patent
by_cell = df_encoded.groupby("Cell_Type")["Time_of_administration_h"].agg(
    n="size", miss_rate=lambda s: s.isna().mean())
print("missing-time rate by cell line:")
print(by_cell[by_cell["n"] >= 200].sort_values("miss_rate", ascending=False).round(2))

print(f"\nsource experiments (Title) total: {df_encoded['Title'].nunique()}")
print(f"experiments that contain missing-time rows: {df_encoded.loc[miss, 'Title'].nunique()}")
print("\nexperiments by number of missing-time rows:")
print(df_encoded.loc[miss, "Title"].value_counts().head(5))

missing-time rate by cell line (>=200 rows):
                                           n  miss_rate
Cell_Type                                              
HepG2                                   1651       0.35
Hep3B                                  12165       0.28
HEK293A                                 1048       0.25
Hela                                    2005       0.24
Primary Cynomolgus Monkey Hepatocytes   4445       0.16
Huh7                                    1602       0.16
Be(2)C cell line                        1966       0.15
Primary human hepatocytes               3015       0.03
Primary mouse hepatocytes                501       0.00
COS7                                    3888       0.00
Human iPSC-derived cortical neurons      314       0.00
Neuro2A cell line                        454       0.00
Non-human hepatocytes                    206       0.00

source experiments (Title) total          : 110
experiments that contain missing-time rows: 18

experiments by num

### Which cell lines survive, and the animal-cell check

Keep animal cells only when they are primary in-vitro cultures. We drop all in-vivo animal rows. The table below checks that this held and shows what survived.

Every survivor that the cleaner recognizes as animal (by a species keyword) is a primary culture. 

"Non-human hepatocytes" is ambiguous (it is animal, has no keyword, and is not labelled primary), so we keep/drop them?

In [47]:
animal_terms = cleaner.animal_terms

counts = df_encoded["Cell_Type"].value_counts()
aud = pd.DataFrame({"rows": counts})

print(f"distinct cell lines after cleaning + encoding: {len(aud)}")
print(aud)

distinct cell lines after cleaning + encoding: 17
                                        rows
Cell_Type                                   
Hep3B                                  12165
Primary Cynomolgus Monkey Hepatocytes   4445
COS7                                    3888
Primary human hepatocytes               3015
Hela                                    2005
Be(2)C cell line                        1966
HepG2                                   1651
Huh7                                    1602
HEK293A                                 1048
Primary mouse hepatocytes                501
Neuro2A cell line                        454
Human iPSC-derived cortical neurons      314
Non-human hepatocytes                    206
Hepa1-6                                  139
Primary Macaque Hepatocytes               22
RT-4                                      19
Human neuroblastoma Kelly cells(DSMZ)     10


### FeatureNormalizer

The two conditions are put on a 0 to 1 scale.

Concentration_norm is log10 of Concentration_nM, then min-max scaled to the range 0 to 1. Doses span picomolar to nanomolar, so the log comes first.

Time_norm is Time_of_administration_h, min-max scaled to the range 0 to 1.

The raw columns are kept as they are, and missing times stay missing.

In [37]:
normalizer = FeatureNormalizer(df_encoded.copy())
df_normalized = normalizer.normalize()
df_normalized[["Concentration_nM", "Concentration_norm",
               "Time_of_administration_h", "Time_norm"]].head()

,Concentration_nM,Concentration_norm,Time_of_administration_h,Time_norm
0,100.0,0.950412,48.0,0.166667
1,100.0,0.950412,48.0,0.166667
2,100.0,0.950412,48.0,0.166667
3,100.0,0.950412,48.0,0.166667
4,100.0,0.950412,48.0,0.166667


In [38]:
for col in ["Concentration_norm", "Time_norm"]:
    s = df_normalized[col]
    print(f"{col} min {s.min():.3f} max {s.max():.3f} NaN {s.isna().sum()}")
    assert s.dropna().between(0, 1).all(), f"{col} escaped [0, 1]"

display(
    df_normalized[["Time_of_administration_h", "Time_norm"]]
    .drop_duplicates()
    .sort_values("Time_of_administration_h")
    .reset_index(drop=True)
)

Concentration_norm min 0.000 max 1.000 NaN 0
Time_norm min 0.000 max 1.000 NaN 6091


,Time_of_administration_h,Time_norm
0,24.0,0.000000
1,40.0,0.111111
2,48.0,0.166667
3,72.0,0.333333
4,168.0,1.000000
5,NaN,NaN


### The final table

The cleaned, encoded and normalized table that feeds the model. The raw columns sit next to the derived ones (Concentration_nM, Concentration_norm, Time_of_administration_h, Time_norm and Cell_Type_One_Hot).

In [48]:
print("Final shape:", df_normalized.shape)
print("Columns:", list(df_normalized.columns))
pd.set_option("display.max_columns", None)
df_normalized.head()

Final shape: (33450, 31)
Columns: ['ID', 'patent_ID', 'Authorization_status', 'Accession_number', 'Target_Gene', 'Gene_ID', 'The_name_of_double_helix', 'Antisense_seqence', 'length_anti_sense_strand', 'Modifications_AntiSense_strand', 'Modification_Types_Antisense_strand', 'Modification_locations_Antisense_strand', 'x', 'Sense_seqence', 'length_sense_strand', 'Modifications_sense_strand', 'Modification_Types_Sense_strand', 'Modification_locations_Sense_strand', 'position_Sense_strand', 'Inhibition', 'SD', 'Cell_Type', 'Concentration', 'Time_of_administration', 'Title', 'Modifications_AntiSense_strand_3_5', 'Concentration_nM', 'Time_of_administration_h', 'Cell_Type_One_Hot', 'Concentration_norm', 'Time_norm']


,ID,patent_ID,Authorization_status,Accession_number,Target_Gene,Gene_ID,The_name_of_double_helix,Antisense_seqence,length_anti_sense_strand,Modifications_AntiSense_strand,Modification_Types_Antisense_strand,Modification_locations_Antisense_strand,x,Sense_seqence,length_sense_strand,Modifications_sense_strand,Modification_Types_Sense_strand,Modification_locations_Sense_strand,position_Sense_strand,Inhibition,SD,Cell_Type,Concentration,Time_of_administration,Title,Modifications_AntiSense_strand_3_5,Concentration_nM,Time_of_administration_h,Cell_Type_One_Hot,Concentration_norm,Time_norm
0,001-01-01-00001-100n-48h-88.00,CN107365771B,Unknown Status,NM_001330729.2,CTNNB1,1499.0,D1,UUAGAAAGCUGAUGGACCAUAACUG,25.0,UUAGAAAGCUGAUGGACCAUAACUG,1*U || 2*U || 3*A || 4*G || 5*A || 6*A || 7*A || 8*G || 9*C || 10*U || 11*G ...,NaN,NaN,CAGUUAUGGUCCAUCAGCUUUCUAA,25.0,mCmAmGmUmUmAmUGGUCCAUCAGC(mU)mUmUmCmUmAmA,1*2'-O-Methylcytidine || 2*2'-O-Methyladenosine || 3*2'-O-Methylguanosine ||...,Sugar,"1,2,3,4,5,6,7,19,20,21,22,23,24,25",88.0,NaN,Hela,100nM,48h,用于抑制CTNNB1靶基因mRNA表达的寡核酸分子及其成套组合物,GUCAAUACCAGGUAGUCGAAAGAUU,100.0,48.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.950412,0.166667
1,001-01-01-00002-100n-48h-90.00,CN107365771B,Unknown Status,NM_001330729.2,CTNNB1,1499.0,D2,UACAAUAGCAGACACCAUCUGAGGA,25.0,UACAAUAGCAGACACCAUCUGAGGA,1*U || 2*A || 3*C || 4*A || 5*A || 6*U || 7*A || 8*G || 9*C || 10*A || 11*G ...,NaN,NaN,UCCUCAGAUGGUGUCUGCUAUUGUA,25.0,mUmCmCmUmCmAmGAUGGUGUCUGC(mU)mAmUmUmGmUmA,1*2'-O-Methyluridine || 2*2'-O-Methylcytidine || 3*2'-O-Methylcytidine || 4*...,Sugar,"1,2,3,4,5,6,7,19,20,21,22,23,24,25",90.0,NaN,Hela,100nM,48h,用于抑制CTNNB1靶基因mRNA表达的寡核酸分子及其成套组合物,AGGAGUCUACCACAGACGAUAACAU,100.0,48.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.950412,0.166667
2,001-01-01-00003-100n-48h-90.00,CN107365771B,Unknown Status,NM_001330729.2,CTNNB1,1499.0,D3,UGAACAAGACGUUGACUUGGAUCUG,25.0,UGAACAAGACGUUGACUUGGAUCUG,1*U || 2*G || 3*A || 4*A || 5*C || 6*A || 7*A || 8*G || 9*A || 10*C || 11*G ...,NaN,NaN,CAGAUCCAAGUCAACGUCUUGUUCA,25.0,mCmAmGmAmUmCmCAAGUCAACGUC(mU)mUmGmUmUmCmA,1*2'-O-Methylcytidine || 2*2'-O-Methyladenosine || 3*2'-O-Methylguanosine ||...,Sugar,"1,2,3,4,5,6,7,19,20,21,22,23,24,25",90.0,NaN,Hela,100nM,48h,用于抑制CTNNB1靶基因mRNA表达的寡核酸分子及其成套组合物,GUCUAGGUUCAGUUGCAGAACAAGU,100.0,48.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.950412,0.166667
3,001-01-01-00004-100n-48h-89.00,CN107365771B,Unknown Status,NM_001330729.2,CTNNB1,1499.0,D4,UUUAGUUGCAGCAUCUGAAAGAUUC,25.0,UUUAGUUGCAGCAUCUGAAAGAUUC,1*U || 2*U || 3*U || 4*A || 5*G || 6*U || 7*U || 8*G || 9*C || 10*A || 11*G ...,NaN,NaN,GAAUCUUUCAGAUGCUGCAACUAAA,25.0,mGmAmAmUmCmUmUUCAGAUGCUGC(mA)mAmCmUmAmAmA,1*2'-O-Methylguanosine || 2*2'-O-Methyladenosine || 3*2'-O-Methyladenosine |...,Sugar,"1,2,3,4,5,6,7,19,20,21,22,23,24,25",89.0,NaN,Hela,100nM,48h,用于抑制CTNNB1靶基因mRNA表达的寡核酸分子及其成套组合物,CUUAGAAAGUCUACGACGUUGAUUU,100.0,48.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.950412,0.166667
4,001-01-01-00005-100n-48h-87.00,CN107365771B,Unknown Status,NM_001330729.2,CTNNB1,1499.0,D5,UUUCGAAUCAAUCCAACAGUAGCCU,25.0,UUUCGAAUCAAUCCAACAGUAGCCU,1*U || 2*U || 3*U || 4*C || 5*G || 6*A || 7*A || 8*U || 9*C || 10*A || 11*A ...,NaN,NaN,AGGCUACUGUUGGAUUGAUUCGAAA,25.0,mAmGmGmCmUmAmCUGUUGGAUUGA(mU)mUmCmGmAmAmA,1*2'-O-Methyladenosine || 2*2'-O-Methylguanosine || 3*2'-O-Methylguanosine |...,Sugar,"1,2,3,4,5,6,7,19,20,21,22,23,24,25",87.0,NaN,Hela,100nM,48h,用于抑制CTNNB1靶基因mRNA表达的寡核酸分子及其成套组合物,UCCGAUGACAACCUAACUAAGCUUU,100.0,48.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.950412,0.166667
